In [6]:
import os
import glob
import numpy as np
import pandas as pd


In [14]:
# where the local coordinate csv files are present
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Global2Local/local"
# where the wing amplitude csv files should be saved
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Wing_Dynamics"

# Creates folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Geometry function 
def get_wing_angles(tip, base):
    # input= tip (x, y, z), base (wing hinge)
    # output= stroke angle(phi), deviation angle(theta)

    # wing vector
    vec = tip - base
    # stroke angle (phi)
    phi = np.degrees(np.arctan2(vec[:, 1], vec[:, 0]))
    # horizontal projection
    hypot_xy = np.sqrt(vec[:, 0]**2 + vec[:, 1]**2)
    # deviation angle (theta)
    theta = np.degrees(np.arctan2(vec[:, 2], hypot_xy))

    return phi, theta 

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_local.csv")))

# (sampling rate/ wing beat frequency)*safety factor
WINDOW_SIZE = 25

for path in csv_files:
    fname = os.path.basename(path)
    trial_name = fname.replace("_local.csv", "")

    print(f"Calculating Wing Power for: {trial_name}")

    df = pd.read_csv(path)

    l_tip  = df[["pt5_X_local", "pt5_Y_local", "pt5_Z_local"]].values
    l_base = df[["pt3_X_local", "pt3_Y_local", "pt3_Z_local"]].values
    r_tip  = df[["pt6_X_local", "pt6_Y_local", "pt6_Z_local"]].values
    r_base = df[["pt4_X_local", "pt4_Y_local", "pt4_Z_local"]].values

    phi_L, theta_L = get_wing_angles(l_tip, l_base)
    phi_R, theta_R = get_wing_angles(r_tip, r_base)

    # amplitude
    amp_L = pd.Series(phi_L).rolling(window=WINDOW_SIZE, center=True).apply(np.ptp)
    amp_R = pd.Series(phi_R).rolling(window=WINDOW_SIZE, center=True).apply(np.ptp)

    kinematics_df = pd.DataFrame({
        "frame": df["frame"],
        "stroke_L": phi_L,
        "amplitude_L": amp_L,
        "stroke_R": phi_R,
        "amplitude_R": amp_R
    })

    # save file
    out_name = f"{trial_name}_wing_kinematics.csv"
    kinematics_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

print(f"\nSuccess! All wing dynamics saved to: {OUTPUT_DIR}")



    


Calculating Wing Power for: Trial2_180rpm
Calculating Wing Power for: Trial2_200rpm
Calculating Wing Power for: Trial3_180rpm
Calculating Wing Power for: Trial4_400rpm
Calculating Wing Power for: Trial5_180rpm
Calculating Wing Power for: Trial5_400rpm
Calculating Wing Power for: Trial7_400rpm

Success! All wing dynamics saved to: ./../../dataFolders/MuscaChasingBeads/Wing_Dynamics
